In [11]:
import pandas as pd
import numpy as np

# 1. Load Data
df = pd.read_csv('vr_hardware_master_dataset.csv')

# 2. Step 1: Data Cleaning
# Drop columns that have no variance or are identifiers not tied to hardware performance
cols_to_drop = ['ScreenBrightness', 'device_id', 'created_at', 'location']
df_clean = df.drop(columns=cols_to_drop, errors='ignore')

# 3. Define the metrics we want to aggregate
# We ignore Timestamp, db_id, room_label, and noise_type for the math
metrics = [
    'TotalUsedMem', 'CpuUtil', 'GpuUtil', 'BatteryMicroAmps', 
    'BatteryTemp', 'BatteryLevel', 'BatteryVoltageMv', 
    'AvgFPS', 'WorstFrameMs', 'BestFrameMs', 'MainThreadMs', 
    'GCAllocRate', 'FrameTimeStdDev', 'CpuClockFreq'
]

# 4. Step 2: Feature Engineering (Aggregation)
# We group by the unique trial ID (db_id) and the labels we want to predict
grouped = df_clean.groupby(['db_id', 'room_label', 'noise_type'])

# We will calculate multiple statistics for every single metric
agg_funcs = {
    metric: ['mean', 'std', 'max', 'min', 'median'] for metric in metrics
}

# Apply the aggregation
df_features = grouped.agg(agg_funcs).reset_index()

# Flatten the hierarchical column names (e.g., 'CpuUtil_mean', 'CpuUtil_std')
df_features.columns = ['_'.join(col).strip('_') for col in df_features.columns.values]

print(f"Original shape: {df.shape}")
print(f"New feature matrix shape: {df_features.shape}")

# Optional: Add a Range column (Max - Min) for high-variance metrics like Jitter
df_features['WorstFrameMs_range'] = df_features['WorstFrameMs_max'] - df_features['WorstFrameMs_min']
df_features['CpuUtil_range'] = df_features['CpuUtil_max'] - df_features['CpuUtil_min']

# Save the ML-ready dataset
df_features.to_csv('ml_ready_features.csv', index=False)
print("Saved aggregated dataset to 'ml_ready_features.csv'")

Original shape: (10087, 22)
New feature matrix shape: (110, 73)
Saved aggregated dataset to 'ml_ready_features.csv'


In [12]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load Data
df = pd.read_csv('vr_hardware_master_dataset.csv')

# 2. Data Cleaning
cols_to_drop = ['ScreenBrightness', 'device_id', 'created_at', 'location', 'rescan']
df_clean = df.drop(columns=cols_to_drop, errors='ignore')

# Identify hardware metrics
metrics = [
    'TotalUsedMem', 'CpuUtil', 'GpuUtil', 'BatteryMicroAmps', 
    'BatteryTemp', 'BatteryLevel', 'BatteryVoltageMv', 
    'AvgFPS', 'WorstFrameMs', 'BestFrameMs', 'MainThreadMs', 
    'GCAllocRate', 'FrameTimeStdDev', 'CpuClockFreq'
]

# 3. Feature Engineering (Aggregation)
grouped = df_clean.groupby(['db_id', 'room_label', 'noise_type'])
agg_funcs = {metric: ['mean', 'std', 'max', 'min', 'median'] for metric in metrics}
df_features = grouped.agg(agg_funcs).reset_index()

# Flatten column names
df_features.columns = ['_'.join(col).strip('_') for col in df_features.columns.values]

# Create custom range features
df_features['WorstFrameMs_range'] = df_features['WorstFrameMs_max'] - df_features['WorstFrameMs_min']
df_features['CpuUtil_range'] = df_features['CpuUtil_max'] - df_features['CpuUtil_min']

# Save the dataset
df_features.to_csv('ml_ready_features.csv', index=False)
print("Saved aggregated features to 'ml_ready_features.csv'")

# 4. Model Setup
y = df_features['room_label']
X = df_features.drop(columns=['db_id', 'room_label', 'noise_type'], errors='ignore')

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# 5. Evaluation
print("--- 5-Fold Cross-Validation Results ---")
cv_scores = cross_val_score(rf_model, X, y, cv=5, scoring='accuracy')
print(f"Mean CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

y_pred = cross_val_predict(rf_model, X, y, cv=5)
print("\n--- Classification Report ---")
print(classification_report(y, y_pred))

Saved aggregated features to 'ml_ready_features.csv'
--- 5-Fold Cross-Validation Results ---
Mean CV Accuracy: 0.5545 (+/- 0.3421)

--- Classification Report ---
              precision    recall  f1-score   support

  conference       0.64      0.60      0.62        30
     hallway       0.70      0.81      0.75        26
     kitchen       0.33      0.19      0.24        26
         lab       0.46      0.61      0.52        28

    accuracy                           0.55       110
   macro avg       0.53      0.55      0.53       110
weighted avg       0.54      0.55      0.54       110



In [13]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import classification_report

# 1. Load the aggregated features
df = pd.read_csv('ml_ready_features.csv')

y = df['room_label']
X = df.drop(columns=['db_id', 'room_label', 'noise_type'], errors='ignore')

# 2. Setup Base Model & Recursive Feature Elimination (RFE)
rf_base = RandomForestClassifier(n_estimators=100, random_state=42)

print("Running RFE to extract the Top 20 features...")
rfe = RFE(estimator=rf_base, n_features_to_select=20, step=2)
rfe.fit(X, y)

# 3. Extract the winning features
selected_features = X.columns[rfe.support_]
X_selected = X[selected_features]

# 4. Train and Evaluate the optimized model
rf_selected = RandomForestClassifier(n_estimators=100, random_state=42)

print("\n--- 5-Fold CV Results (Top 20 Features) ---")
cv_scores = cross_val_score(rf_selected, X_selected, y, cv=5, scoring='accuracy')
print(f"Mean CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

y_pred = cross_val_predict(rf_selected, X_selected, y, cv=5)
print("\n--- Classification Report ---")
print(classification_report(y, y_pred))

# 5. Save the optimized dataset for the next step
df_selected = df[['db_id', 'room_label', 'noise_type'] + list(selected_features)]
df_selected.to_csv('ml_ready_features_top20.csv', index=False)

Running RFE to extract the Top 20 features...

--- 5-Fold CV Results (Top 20 Features) ---
Mean CV Accuracy: 0.6000 (+/- 0.3916)

--- Classification Report ---
              precision    recall  f1-score   support

  conference       0.68      0.57      0.62        30
     hallway       0.62      0.81      0.70        26
     kitchen       0.42      0.31      0.36        26
         lab       0.62      0.71      0.67        28

    accuracy                           0.60       110
   macro avg       0.59      0.60      0.59       110
weighted avg       0.59      0.60      0.59       110



In [14]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load Data
df = pd.read_csv('ml_ready_features_top20.csv')

# Separate into static (0) and noise (1 and 2)
train_df = df[df['noise_type'] == 0]
test_df = df[df['noise_type'] != 0]

print(f"Training on {len(train_df)} static trials...")
print(f"Testing on {len(test_df)} noisy trials...")

# 2. Setup Features and Targets
X_train = train_df.drop(columns=['db_id', 'room_label', 'noise_type'], errors='ignore')
y_train = train_df['room_label']

X_test = test_df.drop(columns=['db_id', 'room_label', 'noise_type'], errors='ignore')
y_test = test_df['room_label']

# 3. Train on Static, Predict on Noise
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
print("\n--- Classification Report (Trained on Static, Tested on Noise) ---")
print(classification_report(y_test, y_pred))

Training on 42 static trials...
Testing on 68 noisy trials...

--- Classification Report (Trained on Static, Tested on Noise) ---
              precision    recall  f1-score   support

  conference       0.93      0.65      0.76        20
     hallway       1.00      0.06      0.12        16
     kitchen       0.17      0.06      0.09        16
         lab       0.34      1.00      0.51        16

    accuracy                           0.46        68
   macro avg       0.61      0.44      0.37        68
weighted avg       0.63      0.46      0.39        68

